# Building an Organizations Collaboration Network using the Dimensions API. 

This notebook shows how to analyse organizations collaboration data so to create an N-degres network, in which the nodes are the organizations working together, and the edges are the publications they have in common. 

PS An earlier version of this work has been described in this blog post too https://www.dimensions.ai/blog/2018/02/building-institutional-collaboration-diagrams-with-the-dimensions-search-api

## Prerequisites: load libraries and log in

In [1]:
# @markdown # Get the API library and login 
# @markdown Click the 'play' button on the left (or shift+enter) after entering your API credentials

username = "adam@uberresearch.com" #@param {type: "string"}
password = "VNzyd1.d8" #@param {type: "string"}
endpoint = "https://app.dimensions.ai" #@param {type: "string"}


!pip install dimcli plotly networkx -U --quiet 
import dimcli
dimcli.login(username, password, endpoint)
dsl = dimcli.Dsl()

#
# load common libraries
import time
import sys
import json
import pandas as pd
from pandas.io.json import json_normalize
from tqdm.notebook import tqdm as progress
import networkx as nx

#
# charts libs
# import plotly_express as px
import plotly.express as px
if not 'google.colab' in sys.modules:
  # make js dependecies local / needed by html exports 
  from plotly.offline import init_notebook_mode
  init_notebook_mode(connected=True)

DimCli v0.6.2.2 - Succesfully connected to <https://app.dimensions.ai> (method: manual login)


## Choose an Organization and a keyword (topic)

For the purpose of this exercise, we will use [grid.4691.a](https://grid.ac/institutes/grid.4691.a) (University of Naples). 

> You can try using a different GRID ID to see how results change, e.g. by [browsing for another GRID organization](https://grid.ac/institutes).


In [2]:
GRIDID = "grid.4691.a" #  #@param {type:"string"}
    
#@markdown The start/end year of publications used to extract patents
YEAR_START = 2000 #@param {type: "slider", min: 1950, max: 2020}
YEAR_END = 2016 #@param {type: "slider", min: 1950, max: 2020}

#@markdown ---
#@markdown A keyword used to filter publications search
TOPIC = "nanotechnology" #@param {type:"string"}

if YEAR_END < YEAR_START:
  YEAR_END = YEAR_START

#
# gen link to Dimensions
#
try:
  gridname = dsl.query(f"""search organizations where id="{GRIDID}" return organizations[name]""", verbose=False).organizations[0]['name']
except:
  gridname = ""
from IPython.core.display import display, HTML
display(HTML('GRID: <a href="{}" title="View selected organization in Dimensions">{} - {} &#x29c9;</a>'.format(dimensions_url(GRIDID), GRIDID, gridname)))
display(HTML('Time period: {} to {}'.format(YEAR_START, YEAR_END)))
display(HTML('Topic: "{}" <br /><br />'.format(TOPIC)))


NameError: name 'dimensions_url' is not defined

## Return the top 10 collaborating institutions

We can use the [publications API](https://docs.dimensions.ai/dsl/data-sources.html#publications) to find the top 10 collaborating institutions based on the parameters above. The function below runs the API query and returns the results in the form of a pandas dataframe, which will make it easier to process the data later. 

A couple of things to note: 

* The resulting dataframe contains two extra columns: `id_from` is the seed institution; `level` is an optional parameter representing the network depth of the query (we'll see later how it is used with recursive querying).
* The query returns 11 records cause the first one is normally the seed GRID (= internal collaborations) which we'll be omitting 
* It'd be easy to add other constraints to the query e.g. specifying research areas via FOR codes, or setting a threshold based on citation counts. The possibilities are endless!  

In [ ]:
def get_collaborators(orgid, level=1, printquery=False):
    if TOPIC:
        TOPIC_CLAUSE = f"""for "{TOPIC}" """
    else:
        TOPIC_CLAUSE = ""
    searchstring = """search publications {}
                       where year in [{}:{}] 
                       and research_orgs.id="{}"
                       return research_orgs
                       limit 11""".format(TOPIC_CLAUSE, YEAR_START, YEAR_END, orgid)
    if printquery: print(searchstring)
    df = dsl.query(searchstring, verbose=False).as_dataframe()
    df['id_from'] = [orgid] * len(df)
    df['level'] = [level] * len(df)
    return df

For example, let's try it out with our GRID:

In [ ]:
get_collaborators(GRIDID, printquery=True)

## Building a collaboration network via repeated queries 

If we think of the organizations collaboration data as a network with nodes and edges, the function above is limited as it lets us obtain only the objects **directly** linked to a GRID. Instead, we want run the same analysis for each GRID in our results, so to generate an N-degrees network where N is chosen by us.  

To this purpose, we can set up a [recursive](https://en.wikipedia.org/wiki/Recursion_(computer_science)) query, which repeats the application of the `get_collaborators` as many times as needed. Note: 
* the `maxlevel` parameter determines how big our network should be (1 = immediate neighbours only, 2 = collaborators of immediate neighbours,e tc..) 
* we pause 1 second at each iteration to avoid hitting the normal API quota (30 per minute)
* goes without saying that calling this function with a high `maxlevel` value will produce a lot of data!


In [ ]:
def looper(seed, maxlevel=1, thislevel=1):
    "Recursive function for building an organization collaboration network"
    collaborators = get_collaborators(seed, thislevel)
    time.sleep(1)
    print("--" * thislevel, seed, " :: level =", thislevel)
    if thislevel < maxlevel:
        gridslist = list(collaborators[collaborators['id'] != GRIDID]['id'])
        extra = [looper(x, maxlevel, thislevel+1) for x in gridslist]
        return collaborators.append(extra)
    else:
        # finally
        return collaborators

Let's try this out with by building a 2-degrees network, which will result in around 100 nodes.   

In [ ]:
collaborators = looper(GRIDID, 2)
# change column order for readability 
collaborators.rename(columns={"id": "id_to"}, inplace=True)
collaborators = collaborators[['id_from', 'id_to', 'level', 'count', 'name', 'acronym', 'city_name', 'state_name', 'country_name', 'latitude', 'longitude', 'linkout',  'types' ]]
collaborators.head()

## Let's visualize the network data

The [pyvis](https://pyvis.readthedocs.io/en/latest/tutorial.html) library allows to create nice [vis.js](https://visjs.org/) network graphs directly from Python. We can use it to build a interactive network visualization showing the organizations collaboration network. 

Comments: 

* Since pyvis produces html outputs that are not compatible with Google Colab notebooks, we use a modified version of it in [dimcli.core.extras](https://github.com/digital-science/dimcli/blob/master/dimcli/core/extras.py) that address this problem 
* The chart colors are straight from [plotly lovely color scales](https://plot.ly/python/builtin-colorscales/). Try changing them!
* Customize the visual language: for example, try changing the way node sizes and colors are being generated so to color-code organizations *countries* or *types* 
* Update the `repulsion` parameter if the chart gets too busy


In [ ]:

from dimcli.core.extras import NetworkViz

# set up dataviz

g = NetworkViz(notebook=True, width="100%", height="800px")
g.toggle_hide_edges_on_drag(False)
g.barnes_hut()
g.repulsion(500)
# g.show_buttons() # in html-standalone mode, this command show viz controls

#
# create nodes and edges
#

# remove duplicates from nodes 
nodes = collaborators.drop_duplicates(subset ="id_to", keep = 'first')
# remove internal collaborations stats 
edges = collaborators[(collaborators['id_to'] != collaborators['id_from'])]

# reuse plotly color palette
palette = px.colors.diverging.Temps


#
# add nodes
#
for index, row in nodes.iterrows():
    
    # calc size based on level
    maxsize = int(nodes['level'].max()) + 1
    if row['id_to'] == GRIDID:
        size = maxsize
    else:
        size = maxsize - row['level']

    # calc color based on level
    if row['id_to'] == GRIDID:
        color = palette[0]
    else:
        color = palette[row['level'] * 2]

    g.add_node(
        n_id = row['id_to'],
        label = row['name'],
        title = f"<h4>{row['name']}<br> - {row['id_to']}</h4>",
        value = size,
        color = color,
        borderWidthSelected = 5,
        shape = "dot",
    )



#
# add edges
#
for index, row in edges.iterrows():
  g.add_edge(row['id_from'], row['id_to'], 
             value=int(row['count'] / 10), 
             label=int(row['count']), 
             arrows="none"
            )


# add tooltips with adjancent links info
neighbor_map = g.get_adj_list() 
for node in g.nodes:
    neigh = neighbor_map[node["id"]]
    labels = [nodes[nodes['id_to'] == x].iloc[0]['name'] for x in neigh]
    node["title"] += "Links:<li>" + "</li><li>".join(labels)
 
    
g.show(f"network_{GRIDID}.html")